# 🤖 CreditIQ — Model Training & Evaluation

In this notebook we:
1. Preprocess the data (clean, engineer features, scale)
2. Apply SMOTE to fix class imbalance
3. Train 3 models: Logistic Regression → Random Forest → XGBoost
4. Evaluate with the right metrics (ROC-AUC, not accuracy!)
5. Explain predictions with SHAP

> **Note:** Run `01_eda.ipynb` first to understand the data.

In [5]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

from sklearn.metrics import (
    roc_auc_score, classification_report,
    confusion_matrix, roc_curve, ConfusionMatrixDisplay
)

from src.preprocess import load_data, preprocess

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('Libraries loaded ✅')

Libraries loaded ✅


## 1. Load & Preprocess Data

In [6]:
df = load_data('../data/cs-training.csv')
X_train, X_test, y_train, y_test, feature_names, scaler = preprocess(df)

print(f'\nFeatures used ({len(feature_names)}):')
for f in feature_names:
    print(f'  - {f}')

FileNotFoundError: [Errno 2] No such file or directory: '../data/cs-training.csv'

## 2. Apply SMOTE

**Why SMOTE?** Our dataset has ~14:1 imbalance. Without fixing this:
- A model that ALWAYS predicts "no default" gets 93.5% accuracy
- But it catches ZERO actual defaults — completely useless for a bank

SMOTE creates synthetic samples of the minority class (defaults) by interpolating between real examples.

In [ ]:
print('Class distribution BEFORE SMOTE:')
print(f'  No Default: {(y_train == 0).sum():,}')
print(f'  Default:    {(y_train == 1).sum():,}')
print(f'  Ratio: {(y_train == 0).sum() / (y_train == 1).sum():.1f}:1')

smote = SMOTE(random_state=42, sampling_strategy=0.3)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print('\nClass distribution AFTER SMOTE:')
print(f'  No Default: {(y_train_bal == 0).sum():,}')
print(f'  Default:    {(y_train_bal == 1).sum():,}')
print(f'  Ratio: {(y_train_bal == 0).sum() / (y_train_bal == 1).sum():.1f}:1')

## 3. Train Three Models

We train from simplest to most complex:
- **Logistic Regression** — linear baseline
- **Random Forest** — ensemble of trees
- **XGBoost** — gradient boosting (usually best)

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest':       RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1),
    'XGBoost':             XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                                          subsample=0.8, colsample_bytree=0.8,
                                          random_state=42, eval_metric='auc', verbosity=0)
}

trained = {}
results = {}

for name, model in models.items():
    print(f'Training {name}...', end=' ')
    model.fit(X_train_bal, y_train_bal)
    proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba)
    print(f'ROC-AUC = {auc:.4f} ✅')
    trained[name] = model
    results[name] = {'auc': auc, 'proba': proba}

## 4. Why ROC-AUC and NOT Accuracy?

> If you use accuracy on imbalanced data, you get a useless model that looks great.

In [ ]:
# Demonstrate why accuracy is deceptive here
dummy_pred = np.zeros(len(y_test))  # always predict 'no default'
dummy_acc  = (dummy_pred == y_test).mean()
dummy_auc  = roc_auc_score(y_test, dummy_pred)

print('Naive model (always predict 0):')
print(f'  Accuracy: {dummy_acc:.1%}  ← looks great!')
print(f'  ROC-AUC:  {dummy_auc:.4f}  ← reveals it\'s useless')
print()
print('XGBoost:')
xgb_proba = results['XGBoost']['proba']
xgb_pred = (xgb_proba >= 0.5).astype(int)
print(f'  Accuracy: {(xgb_pred == y_test).mean():.1%}')
print(f'  ROC-AUC:  {results["XGBoost"]["auc"]:.4f}')

## 5. ROC Curves — Visual Model Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

colors = ['#3498db', '#e67e22', '#e74c3c']
for (name, data), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, data['proba'])
    ax.plot(fpr, tpr, label=f"{name} (AUC = {data['auc']:.3f})", color=color, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.500)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Model Comparison', fontweight='bold')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Confusion Matrix (Best Model)

In [ ]:
best_model = trained['XGBoost']
best_proba = results['XGBoost']['proba']

# Try different thresholds — 0.5 is not always optimal
threshold = 0.3  # lower threshold = catch more defaults (higher recall, lower precision)
y_pred = (best_proba >= threshold).astype(int)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, ax=ax,
    display_labels=['No Default', 'Default'],
    cmap='Blues'
)
ax.set_title(f'Confusion Matrix (threshold={threshold})', fontweight='bold')
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=['No Default', 'Default']))
print('Note: For banks, recall on Default class is critical.')
print('A missed default (false negative) costs much more than a wrong rejection.')

## 7. Feature Importance

In [ ]:
importances = pd.Series(best_model.feature_importances_, index=feature_names).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind='barh', ax=ax, color='#3498db', edgecolor='white')
ax.set_title('XGBoost Feature Importances', fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

## 8. SHAP Explainability

Feature importance tells you which features matter globally.  
**SHAP tells you WHY a specific prediction was made.**

This is what makes a model *explainable* — critical for finance/healthcare applications.

In [ ]:
# Compute SHAP values (using a sample for speed)
X_sample = X_test[:1000]

explainer   = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_sample)

print('SHAP values computed ✅')
print(f'Shape: {shap_values.shape}  (1 SHAP value per feature per sample)')

In [ ]:
# Global SHAP summary — which features drive default predictions most?
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_sample, feature_names=feature_names, show=False)
plt.title('SHAP Summary Plot — Global Feature Impact', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

# How to read this:
# Each dot = one sample
# Position on X axis = SHAP value (how much this feature pushed the prediction)
# Color = feature value (red = high, blue = low)
# → High RevolvingUtilization (red) pushes prediction toward DEFAULT (positive SHAP)

In [ ]:
# Explain a SINGLE prediction
sample_idx = 5  # Change this to look at different applicants

print(f'Applicant #{sample_idx} — Actual: {"DEFAULT" if y_test.iloc[sample_idx] == 1 else "NO DEFAULT"}')
print(f'Model prediction probability: {best_proba[sample_idx]:.1%}')

shap.force_plot(
    explainer.expected_value,
    shap_values[sample_idx],
    X_sample[sample_idx],
    feature_names=feature_names,
    matplotlib=True
)
plt.title(f'Why did the model predict {best_proba[sample_idx]:.1%} default risk?')
plt.tight_layout()
plt.show()

## 9. Save the Best Model

In [4]:
import os, json
os.makedirs('../models', exist_ok=True)

joblib.dump(best_model, '../models/best_model.pkl')
joblib.dump(scaler,     '../models/scaler.pkl')
joblib.dump(feature_names, '../models/feature_names.pkl')
joblib.dump(explainer,  '../models/explainer.pkl')
np.save('../models/shap_values.npy', shap_values)
np.save('../models/X_sample.npy', X_sample)

# Save results summary for the Streamlit app
results_summary = {name: {'auc': round(data['auc'], 4)} for name, data in results.items()}
with open('../models/results.json', 'w') as f:
    json.dump({'results': results_summary, 'best_model': 'XGBoost'}, f, indent=2)

print('All artifacts saved to models/ ✅')
print('\nNow run: streamlit run app.py')

NameError: name 'best_model' is not defined